# Run CycleGAN training on Google Colab

This notebook mounts Google Drive, validates paths, installs small dependencies if missing, and runs Pipeline/Preprocessing/GAN_train_simple.py.

Assumptions: you uploaded your data folder contents into `MyDrive/cobas/` (so files like `o_synced.mp4` and `t_synced.mp4` are in `MyDrive/cobas/`). Also place the trainer script at `MyDrive/cobas/Pipeline/Preprocessing/GAN_train_simple.py`. If the script is not present in Drive, the notebook will show an error and instructions.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install lightweight Python packages needed by the script if missing
# We avoid force-installing torch as Colab usually provides a suitable version.
import sys
import subprocess
import importlib
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
# Ensure OpenCV and Pillow are available
for pkg in ('opencv-python', 'pillow'):
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except Exception:
        print(f'Installing {pkg}...')
        pip_install(pkg)
# Check torch availability and cuda status
try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
except Exception as e:
    print('torch not importable; installing torch and torchvision. This may take a few minutes...')
    pip_install('torch')
    pip_install('torchvision')
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())


In [ ]:
# Configure paths and verify files.
import os
DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_ROOT = os.path.join(DRIVE_ROOT, 'cobas')
GAN_SCRIPT = os.path.join(PROJECT_ROOT, 'Pipeline', 'Preprocessing', 'GAN_train_simple.py')
DATA_ROOT = PROJECT_ROOT  # where you placed the data folder contents
print('Expected project root on Drive:', PROJECT_ROOT)
print('
Checking for key files...')
print('GAN script:', GAN_SCRIPT, '->', 'FOUND' if os.path.exists(GAN_SCRIPT) else 'MISSING')
# check sample video files (common names from the repo listing)
opt_v = os.path.join(DATA_ROOT, 'o_synced.mp4')
th_v = os.path.join(DATA_ROOT, 't_synced.mp4')
print('optical video:', opt_v, '->', 'FOUND' if os.path.exists(opt_v) else 'MISSING')
print('thermal video:', th_v, '->', 'FOUND' if os.path.exists(th_v) else 'MISSING')
print('
If files are missing, upload them to the Drive path above or adjust the paths below.')


In [ ]:
# Run the training script.
# Edit these variables if you want different behavior (epochs, batch size, work dir, paths).
import os
EPOCHS = 10  # reduce for quick tests; increase as needed
BATCH_SIZE = 4
NUM_WORKERS = 2  # set 0 if multiprocessing causes issues on Colab
WORK_DIR = os.path.join(PROJECT_ROOT, 'gan_run')
OPTICAL_VIDEO = os.path.join(DATA_ROOT, 'o_synced.mp4')
THERMAL_VIDEO = os.path.join(DATA_ROOT, 't_synced.mp4')
# If you prefer to use frame-folders instead of videos, set OPTICAL_VIDEO/THERMAL_VIDEO to Noneif not os.path.exists(GAN_SCRIPT):
    raise FileNotFoundError(f'GAN script not found at {GAN_SCRIPT}. Please upload it to Drive at this path.')
# Build command
cmd = [
    'python3', GAN_SCRIPT,
    '--optical-video', OPTICAL_VIDEO,
    '--thermal-video', THERMAL_VIDEO,
    '--work-dir', WORK_DIR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]
print('Running:')
print(' '.join(cmd))
# Execute the training script (streams output into the notebook)
os.execvp(cmd[0], cmd)
